## Stephen WAN Hacking 3 - another layer deeper
- Ok now let me try to get into the model itself

In [ ]:
# Copyright 2024-2025 The Alibaba Wan Team Authors. All rights reserved.
import argparse
from datetime import datetime
import logging
import os
import sys
import warnings

warnings.filterwarnings('ignore')

import torch, random
import torch.distributed as dist
from PIL import Image

import wan
from wan.configs import WAN_CONFIGS, SIZE_CONFIGS, MAX_AREA_CONFIGS, SUPPORTED_SIZES
from wan.utils.prompt_extend import DashScopePromptExpander, QwenPromptExpander
from wan.utils.utils import cache_video, cache_image, str2bool


```
python generate.py  --task t2v-1.3B --size 832*480 --ckpt_dir ./Wan2.1-T2V-1.3B --offload_model True --t5_cpu --sample_shift 8 --sample_guide_scale 6 --prompt "Two anthropomorphic cats in comfy boxing gear and bright gloves fight intensely on a spotlighted stage."
```

In [ ]:
EXAMPLE_PROMPT = {
    "t2v-1.3B": {
        "prompt": "Two anthropomorphic cats in comfy boxing gear and bright gloves fight intensely on a spotlighted stage.",
    },
    "t2v-14B": {
        "prompt": "Two anthropomorphic cats in comfy boxing gear and bright gloves fight intensely on a spotlighted stage.",
    },
    "t2i-14B": {
        "prompt": "一个朴素端庄的美人",
    },
    "i2v-14B": {
        "prompt":
            "Summer beach vacation style, a white cat wearing sunglasses sits on a surfboard. The fluffy-furred feline gazes directly at the camera with a relaxed expression. Blurred beach scenery forms the background featuring crystal-clear waters, distant green hills, and a blue sky dotted with white clouds. The cat assumes a naturally relaxed posture, as if savoring the sea breeze and warm sunlight. A close-up shot highlights the feline's intricate details and the refreshing atmosphere of the seaside.",
        "image":
            "examples/i2v_input.JPG",
    },
}

In [ ]:
def sinusoidal_embedding_1d(dim, position):
    # preprocess
    assert dim % 2 == 0
    half = dim // 2
    position = position.type(torch.float64)

    # calculation
    sinusoid = torch.outer(
        position, torch.pow(10000, -torch.arange(half).to(position).div(half)))
    x = torch.cat([torch.cos(sinusoid), torch.sin(sinusoid)], dim=1)
    return x

In [ ]:
def _validate_args(args):
    # Basic check
    assert args.ckpt_dir is not None, "Please specify the checkpoint directory."
    assert args.task in WAN_CONFIGS, f"Unsupport task: {args.task}"
    assert args.task in EXAMPLE_PROMPT, f"Unsupport task: {args.task}"

    # The default sampling steps are 40 for image-to-video tasks and 50 for text-to-video tasks.
    if args.sample_steps is None:
        args.sample_steps = 40 if "i2v" in args.task else 50

    if args.sample_shift is None:
        args.sample_shift = 5.0
        if "i2v" in args.task and args.size in ["832*480", "480*832"]:
            args.sample_shift = 3.0

    # The default number of frames are 1 for text-to-image tasks and 81 for other tasks.
    if args.frame_num is None:
        args.frame_num = 1 if "t2i" in args.task else 81

    # T2I frame_num check
    if "t2i" in args.task:
        assert args.frame_num == 1, f"Unsupport frame_num {args.frame_num} for task {args.task}"

    args.base_seed = args.base_seed if args.base_seed >= 0 else random.randint(
        0, sys.maxsize)
    # Size check
    assert args.size in SUPPORTED_SIZES[
        args.
        task], f"Unsupport size {args.size} for task {args.task}, supported sizes are: {', '.join(SUPPORTED_SIZES[args.task])}"

In [ ]:
# Set up default arguments as variables for easy modification
task = "t2v-1.3B"
size = "832*480"
frame_num = None
ckpt_dir = './Wan2.1-T2V-1.3B'
offload_model = True
ulysses_size = 1
ring_size = 1
t5_fsdp = False
t5_cpu = True
dit_fsdp = False
save_file = None
prompt = "Two anthropomorphic cats in comfy boxing gear and bright gloves fight intensely on a spotlighted stage."
use_prompt_extend = False
prompt_extend_method = "local_qwen"
prompt_extend_model = None
prompt_extend_target_lang = "zh"
base_seed = -1
image = None
sample_solver = "unipc"
sample_steps = None
sample_shift = 8
sample_guide_scale = 6.0

# Create args object as a SimpleNamespace (similar to argparse.Namespace)
from types import SimpleNamespace
args = SimpleNamespace(
    task=task, size=size, frame_num=frame_num, ckpt_dir=ckpt_dir, offload_model=offload_model,
    ulysses_size=ulysses_size, ring_size=ring_size, t5_fsdp=t5_fsdp, t5_cpu=t5_cpu, 
    dit_fsdp=dit_fsdp, save_file=save_file, prompt=prompt, use_prompt_extend=use_prompt_extend,
    prompt_extend_method=prompt_extend_method, prompt_extend_model=prompt_extend_model,
    prompt_extend_target_lang=prompt_extend_target_lang, base_seed=base_seed, image=image,
    sample_solver=sample_solver, sample_steps=sample_steps, sample_shift=sample_shift,
    sample_guide_scale=sample_guide_scale
)

# Optional: Validate args (you can uncomment and modify as needed)
# def validate_args(args):
#     # Add validation logic here
#     pass
# validate_args(args)

In [ ]:
_validate_args(args)

In [ ]:
args.task

In [ ]:
# def generate(args):
rank = int(os.getenv("RANK", 0))
world_size = int(os.getenv("WORLD_SIZE", 1))
local_rank = int(os.getenv("LOCAL_RANK", 0))
device = local_rank

In [ ]:
device, world_size, local_rank, rank

In [ ]:
cfg = WAN_CONFIGS[args.task]

In [ ]:
cfg

In [ ]:
args.ulysses_size

In [ ]:
dist.is_initialized()

In [ ]:
#This is probably one to break apart. 
wan_t2v = wan.WanT2V(
    config=cfg,
    checkpoint_dir=args.ckpt_dir,
    device_id=device,
    rank=rank,
    t5_fsdp=args.t5_fsdp,
    dit_fsdp=args.dit_fsdp,
    use_usp=(args.ulysses_size > 1 or args.ring_size > 1),
    t5_cpu=args.t5_cpu,
)

### Let's get inside the wan class here

```
video = wan_t2v.generate(
    args.prompt,
    size=SIZE_CONFIGS[args.size],
    frame_num=args.frame_num,
    shift=args.sample_shift,
    sample_solver=args.sample_solver,
    sampling_steps=args.sample_steps,
    guide_scale=args.sample_guide_scale,
    seed=args.base_seed,
    offload_model=args.offload_model)
```


In [ ]:
import gc
import logging
import math
import os
import random
import sys
import types
from contextlib import contextmanager
from functools import partial

import torch
import torch.cuda.amp as amp
import torch.distributed as dist
from tqdm import tqdm

# from .distributed.fsdp import shard_model
sys.path.append('wan')
from modules.model import WanModel
from modules.t5 import T5EncoderModel
from modules.vae import WanVAE
from utils.fm_solvers import (FlowDPMSolverMultistepScheduler,
                               get_sampling_sigmas, retrieve_timesteps)
from utils.fm_solvers_unipc import FlowUniPCMultistepScheduler

In [ ]:
input_prompt=args.prompt
size=SIZE_CONFIGS[args.size]
frame_num=args.frame_num
shift=args.sample_shift
sample_solver=args.sample_solver
sampling_steps=args.sample_steps
guide_scale=args.sample_guide_scale
n_prompt=""
seed=args.base_seed
offload_model=args.offload_model

In [ ]:
F = frame_num

In [ ]:
F

In [ ]:
target_shape = (wan_t2v.vae.model.z_dim, (F - 1) // wan_t2v.vae_stride[0] + 1,
                size[1] // wan_t2v.vae_stride[1],
                size[0] // wan_t2v.vae_stride[2])

In [ ]:
target_shape #Like...chanels, time, height, width?

In [ ]:
seq_len = math.ceil((target_shape[2] * target_shape[3]) /
                    (wan_t2v.patch_size[1] * wan_t2v.patch_size[2]) *
                    target_shape[1] / wan_t2v.sp_size) * wan_t2v.sp_size

In [ ]:
seq_len #Hmm something unrolled here?

In [ ]:
if n_prompt == "":
    n_prompt = wan_t2v.sample_neg_prompt

In [ ]:
n_prompt

'Gorgeous colors, overexposed, static, blurred details, subtitles, style, artwork, painting, picture, still, overall gray, worst quality, low quality, JPEG compression residue, ugly, incomplete, extra fingers, poorly drawn hands, poorly drawn faces, deformed, disfigured, deformed limbs, fused fingers, still picture, cluttered background, three legs, many people in the background, walking backwards

In [ ]:
seed = seed if seed >= 0 else random.randint(0, sys.maxsize)

In [ ]:
seed

In [ ]:
seed_g = torch.Generator(device=wan_t2v.device)

In [ ]:
seed_g.manual_seed(seed)

In [ ]:
seed_g

In [ ]:
wan_t2v.t5_cpu

In [ ]:
context = wan_t2v.text_encoder([input_prompt], torch.device('cpu')) #Damn this is slow -> i guess it is on CPU!

In [ ]:
context

In [ ]:
len(context)

In [ ]:
context[0].shape

In [ ]:
input_prompt

Ok cool -> the prompt must be 28 tokens. 

In [ ]:
context_null = wan_t2v.text_encoder([n_prompt], torch.device('cpu'))

In [ ]:
context_null[0].shape

In [ ]:
context = [t.to(wan_t2v.device) for t in context] #Just shipping to device, got it. 
context_null = [t.to(wan_t2v.device) for t in context_null]

In [ ]:
wan_t2v.device

In [ ]:
#Ah man I love this!
noise = [
    torch.randn(
        target_shape[0],
        target_shape[1],
        target_shape[2],
        target_shape[3],
        dtype=torch.float32,
        device=wan_t2v.device,
        generator=seed_g)
]

In [ ]:
len(noise)

In [ ]:
noise[0].shape

In [ ]:
sample_solver

```
    def forward(
        self,
        x,
        t,
        context,
        seq_len,
        clip_fea=None,
        y=None,
```


In [ ]:
@contextmanager
def noop_no_sync():
    yield

no_sync = getattr(wan_t2v.model, 'no_sync', noop_no_sync)

# evaluation mode
with amp.autocast(dtype=wan_t2v.param_dtype), torch.no_grad(), no_sync():
    sample_scheduler = FlowUniPCMultistepScheduler(
                    num_train_timesteps=wan_t2v.num_train_timesteps,
                    shift=1,
                    use_dynamic_shifting=False)
    sample_scheduler.set_timesteps(
        sampling_steps, device=wan_t2v.device, shift=shift)
    timesteps = sample_scheduler.timesteps

    latents = noise #Fun line

    arg_c = {'context': context, 'seq_len': seq_len}
    arg_null = {'context': context_null, 'seq_len': seq_len}

    for _, t in enumerate(tqdm(timesteps[:2])): #timesteps)):
        latent_model_input = latents
        timestep = [t]
    
        timestep = torch.stack(timestep)
    
        wan_t2v.model.to(wan_t2v.device)
        # noise_pred_cond = wan_t2v.model(
        #     latent_model_input, t=timestep, **arg_c)[0]

        ## --- Break into model forward pass here --- ## 
        x=latent_model_input
        t=timestep
        context=arg_c['context']
        seq_len=arg_c['seq_len']

        # embeddings
        # Ok in this first step we go from torch.Size([16, 21, 60, 104]) to [1, 1536, 21, 30, 52])
        # Ok cool yeah this makes a lot of sense, very similiar to what CogVideoX folks did
        # This is Conv3d layer. 2 by 2 by 16 patches go to 1536 length vectors
        # are they touching the time dimension at all here? No? 
        # wan_t2v.model.patch_embedding
        # Conv3d(16, 1536, kernel_size=(1, 2, 2), stride=(1, 2, 2))
        # wan_t2v.model.patch_embedding.weight.shape
        # torch.Size([1536, 16, 1, 2, 2])
        # yeah so I think that temporal stride of 1 means no. 
        x = [wan_t2v.model.patch_embedding(u.unsqueeze(0)) for u in x] #torch.Size([1, 1536, 21, 30, 52])

        grid_sizes = torch.stack(
            [torch.tensor(u.shape[2:], dtype=torch.long) for u in x]) #tensor([[21, 30, 52]])

        # Ok pretty sure this one is unrolling time, height, and width. So it's kinda like 1536 features each of the full 
        # compressed video size. It's also 2x2x21 patches mapped to emvedding vectors of length 1536. How many 2x2 patches
        # are there in the latents? well, there's 30*52*21=32760. 
        x = [u.flatten(2).transpose(1, 2) for u in x] #torch.Size([1, 32760, 1536]) 
        seq_lens = torch.tensor([u.size(1) for u in x], dtype=torch.long) #tensor([32760])
        assert seq_lens.max() <= seq_len
        # print(seq_len - x[0].size(1))
        x = torch.cat([
            torch.cat([u, u.new_zeros(1, seq_len - u.size(1), u.size(2))],
                      dim=1) for u in x
        ])        ##torch.Size([1, 32760, 1536]) -> Ok I don't think this step is doing anything in this configuration. seq_len - x[0].size(1)=0


        # time embeddings - oh right this is "diffusion time" not "video time" - got it
        with amp.autocast(dtype=torch.float32):
            e = wan_t2v.model.time_embedding(
                sinusoidal_embedding_1d(wan_t2v.model.freq_dim, t).float()) #torch.Size([1, 1536])
            e0 = wan_t2v.model.time_projection(e).unflatten(1, (6, wan_t2v.model.dim)) #torch.Size([1, 6, 1536])
            assert e.dtype == torch.float32 and e0.dtype == torch.float32

        # context
        # context[0].shape = torch.Size([28, 4096])
        #torch.Size([28, 4096]) -> torch.Size([1, 512, 1536]) Interesting.
        #So below were processing through 2 linear layers and a GELU. Individual tokens are lost. Hmmm. 
        context_lens = None
        context_copy = wan_t2v.model.text_embedding( #SW renaming context to context_copy here
            torch.stack([
                torch.cat(
                    [u, u.new_zeros(wan_t2v.model.text_len - u.size(0), u.size(1))])
                for u in context
            ]))  


        # arguments
        kwargs = dict(
            e=e0,
            seq_lens=seq_lens.to('cuda'),
            grid_sizes=grid_sizes.to('cuda'),
            freqs=wan_t2v.model.freqs.to('cuda'),
            context=context_copy,
            context_lens=context_lens)

        for block in wan_t2v.model.blocks: #30 layers
            x = block(x, **kwargs) #Hmm do I need to poke around in here? Maybe if i want to deep dive cross attention - which maybe I do?
        #x.shape=torch.Size([1, 32760, 1536]) 

        # head
        x = wan_t2v.model.head(x, e)
        #x.shape=torch.Size([1, 32760, 64]) 
        #hmm interesting, why is the head dropping down to 64? Why 64? What are we doing again lol? Shouldn't it be 16? Oh wait 2x2? Is that why?
        
        # unpatchify
        x = wan_t2v.model.unpatchify(x, grid_sizes)
        # x[0].shape=torch.Size([16, 21, 60, 104])  32760*64=2,096,640=2,096,640=16*21*60*104 -> got it!

        out_tmp=[u.float() for u in x]
        noise_pred_cond=out_tmp[0]

        ## ---- end opening up model ---- ## 
        
        noise_pred_uncond = wan_t2v.model(
            latent_model_input, t=timestep, **arg_null)[0]
    
        noise_pred = noise_pred_uncond + guide_scale * ( #Woah what a cool/simpel line here - ah very close to Ho's classifier free guidance!
            noise_pred_cond - noise_pred_uncond)
    
        temp_x0 = sample_scheduler.step( #Hmm not really sure what's happening here. Might be plumbing-y. 
            noise_pred.unsqueeze(0),
            t,
            latents[0].unsqueeze(0),
            return_dict=False,
            generator=seed_g)[0]
        latents = [temp_x0.squeeze(0)]

    x0 = latents
    if offload_model:
        wan_t2v.model.cpu()
        torch.cuda.empty_cache()
    if wan_t2v.rank == 0:
        videos = wan_t2v.vae.decode(x0)

In [ ]:
del noise, latents
del sample_scheduler
if offload_model:
    gc.collect()
    torch.cuda.synchronize()
if dist.is_initialized():
    dist.barrier()

In [ ]:
len(videos)

In [ ]:
videos[0].shape

In [ ]:
formatted_time = datetime.now().strftime("%Y%m%d_%H%M%S")
formatted_prompt = args.prompt.replace(" ", "_").replace("/",
                                                         "_")[:50]
suffix = '.png' if "t2i" in args.task else '.mp4'
args.save_file = f"{args.task}_{args.size.replace('*','x') if sys.platform=='win32' else args.size}_{args.ulysses_size}_{args.ring_size}_{formatted_prompt}_{formatted_time}" + suffix

In [ ]:
cache_video(
    tensor=videos[0][None],
    save_file=args.save_file,
    fps=cfg.sample_fps,
    nrow=1,
    normalize=True,
    value_range=(-1, 1))

In [ ]:
args.save_file

- DOPE!

---

This section of `WanModel` init is pretty interesting: 

```
        self.patch_embedding = nn.Conv3d(
            in_dim, dim, kernel_size=patch_size, stride=patch_size)
        self.text_embedding = nn.Sequential(
            nn.Linear(text_dim, dim), nn.GELU(approximate='tanh'),
            nn.Linear(dim, dim))

        self.time_embedding = nn.Sequential(
            nn.Linear(freq_dim, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.time_projection = nn.Sequential(nn.SiLU(), nn.Linear(dim, dim * 6))
```

In [ ]:
videos

In [ ]:
x[0].shape

In [ ]:
offload_model

In [ ]:
x.device

In [ ]:
t.device

In [ ]:
len(wan_t2v.model.blocks)

In [ ]:
wan_t2v.model.device

In [ ]:
        # kwargs = dict(
        #     e=e0,
        #     seq_lens=seq_lens,
        #     grid_sizes=grid_sizes,
        #     freqs=wan_t2v.model.freqs,
        #     context=context_copy,
        #     context_lens=context_lens)

In [ ]:
e0.device

In [ ]:
seq_lens.device

In [ ]:
grid_sizes.device

In [ ]:
wan_t2v.model.freqs.device

In [ ]:
context_copy.device

In [ ]:
context_lens

In [ ]:
kwargs

In [ ]:
context.shape

In [ ]:
clip_fea

In [ ]:
context[0].shape

In [ ]:
len(context)

In [ ]:
e.shape

In [ ]:
e0.shape

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.plot(e.detach().cpu().numpy().ravel())

In [ ]:
len(x)

In [ ]:
x[0].shape

In [ ]:
seq_lens

In [ ]:
x.shape

In [ ]:
seq_len - x.size(1)

In [ ]:
grid_sizes

In [ ]:
x[0].shape

In [ ]:
wan_t2v.model.patch_embedding

In [ ]:
wan_t2v.model.patch_embedding.weight.shape

In [ ]:
cfg

In [ ]:
args

- Man this seems pretty simple! Maybe simpler than CogVideoX
- Maybe there's a big layer lurking below

In [ ]:
latent_model_input[0].shape

In [ ]:
arg_c['context'][0].shape

- Ok yeah, how does the model get these two tensors to play together? That's an important question.
- Yeah so this bit is definitely interesting, I need to get at least one layer deeper I think and see how the model's forward ish handles wrangling these tensors - I'm kinda envisisioning getting to the bottom-ish, and then "instrumenting" the generation process kinda like I did with CogVideoX. Certrainly would be cool to see accross various prompts/prompt ablations. 

In [ ]:
timestep

In [ ]:
noise_pred_cond.shape

In [ ]:
noise_pred_uncond.shape

In [ ]:
noise_pred.shape

In [ ]:
guide_scale

In [ ]:
temp_x0.shape

In [ ]:
arg_c

In [ ]:
del noise, latents
del sample_scheduler
if offload_model:
    gc.collect()
    torch.cuda.synchronize()
if dist.is_initialized():
    dist.barrier()

In [ ]:
timesteps

In [ ]:
sample_scheduler

In [ ]:
sample_scheduler

In [ ]:
timesteps

---

In [ ]:
SIZE_CONFIGS[args.size]

In [ ]:
args.frame_num

In [ ]:
args.sample_shift

In [ ]:
args.sample_solver

In [ ]:
args.sample_steps

In [ ]:
args.sample_guide_scale

In [ ]:
args.base_seed

In [ ]:
args.offload_model

In [ ]:
video = wan_t2v.generate(
    args.prompt,
    size=SIZE_CONFIGS[args.size],
    frame_num=args.frame_num,
    shift=args.sample_shift,
    sample_solver=args.sample_solver,
    sampling_steps=args.sample_steps,
    guide_scale=args.sample_guide_scale,
    seed=args.base_seed,
    offload_model=args.offload_model)

In [ ]:
args.save_file

In [ ]:
formatted_time = datetime.now().strftime("%Y%m%d_%H%M%S")
formatted_prompt = args.prompt.replace(" ", "_").replace("/",
                                                         "_")[:50]
suffix = '.png' if "t2i" in args.task else '.mp4'
args.save_file = f"{args.task}_{args.size.replace('*','x') if sys.platform=='win32' else args.size}_{args.ulysses_size}_{args.ring_size}_{formatted_prompt}_{formatted_time}" + suffix

In [ ]:
cache_video(
    tensor=video[None],
    save_file=args.save_file,
    fps=cfg.sample_fps,
    nrow=1,
    normalize=True,
    value_range=(-1, 1))

In [ ]:
args.save_file